# Vidal LSTM Transfer Learning: Panasonic → CMU eVTOL

Experiment entrypoint and audit surface for the Vidal method. Repository modules own the model, CMU adapter, resampling, window construction, normalization, and metrics. This notebook does not retrain Vidal Case 1–4 and does not invent an unfrozen CMU supervised-adaptation protocol.

# 0. Status and Experiment Manifest

- Paper: Vidal et al. (2019)
- Method: LSTM SOC estimator with supervised transfer learning
- Canonical source: Panasonic NCR18650PF, 25 °C, Case 1
- Target: CMU Sony-Murata VTC6 eVTOL
- Target reference labels: used only for supervised adaptation/evaluation
- Pseudo labels: not used

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import platform
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml
from IPython.display import display

RUN_SOURCE_TRAINING = False
RUN_ZERO_SHOT_SMOKE_TEST = True
RUN_FULL_ZERO_SHOT = False
RUN_SUPERVISED_ADAPTATION = False
WRITE_RESULTS = False
SEED = 42

def find_repo_root(start):
    for path in (start.resolve(), *start.resolve().parents):
        if (path / "pyproject.toml").is_file() and (path / "src/battery_benchmark").is_dir():
            return path
    raise RuntimeError("Run this notebook from inside battery-soc-transfer-benchmark")

REPO_ROOT = find_repo_root(Path.cwd())
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / "src"))

from battery_benchmark.datasets.cmu_evtol.vidal_adapter import (
    ADAPTER_VERSION,
    adapt_cmu_dynamic_soc,
    build_cmu_vidal_windows,
    load_cmu_dynamic_soc_for_vidal,
    transform_cmu_records,
)
from battery_benchmark.evaluation.metrics import compute_metrics
from battery_benchmark.preprocessing.normalization import MinMaxNormalizer
from battery_benchmark.reproductions.vidal_2019.model import VidalLSTM
from battery_benchmark.reproductions.vidal_2019.windows import FEATURES
from battery_benchmark.training.checkpointing import load_checkpoint

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GIT_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_ROOT, text=True).strip()
NOW_UTC = datetime.now(timezone.utc).isoformat()

EXTERNAL_ARTIFACT_ROOT = os.environ.get("VIDAL_ARTIFACT_ROOT")
ARTIFACT_ROOTS = [REPO_ROOT / "outputs"]
if EXTERNAL_ARTIFACT_ROOT:
    ARTIFACT_ROOTS.insert(0, Path(EXTERNAL_ARTIFACT_ROOT).expanduser())

CMU_BASE_ROOT = REPO_ROOT / "outputs/datasets/cmu_evtol/dynamic_capacity_soc_development_v1"
CMU_EXTENSION_ROOT = REPO_ROOT / "outputs/datasets/cmu_evtol/dynamic_capacity_soc_development_v1_extension4"
CMU_INVENTORY_PATH = REPO_ROOT / "outputs/datasets/cmu_evtol/vidal_adapter_v1/core7_inventory.csv"
REPOSITORY_CASE1_CONFIG = REPO_ROOT / "configs/reproductions/vidal_2019/vidal_case1_panasonic.yaml"
RESULT_ROOT = REPO_ROOT / "outputs/benchmarks/panasonic_to_cmu_evtol/vidal"

def discover_canonical_run(roots):
    candidates = []
    for root in roots:
        if root.exists():
            candidates.extend(root.rglob("checkpoints/epoch_515.pt"))
    for checkpoint in sorted(set(candidates), key=lambda p: p.stat().st_mtime, reverse=True):
        run_dir = checkpoint.parent.parent
        scaler = run_dir / "scaler.json"
        config = run_dir / "resolved_config.yaml"
        metadata = run_dir / "run_metadata.json"
        if not all(path.is_file() for path in (scaler, config, metadata)):
            continue
        try:
            payload = torch.load(checkpoint, map_location="cpu", weights_only=False)
            resolved = yaml.safe_load(config.read_text()) or {}
            run_meta = json.loads(metadata.read_text())
        except Exception:
            continue
        checkpoint_cfg = payload.get("config", {})
        dataset = resolved.get("dataset", checkpoint_cfg.get("dataset", run_meta.get("dataset")))
        if (
            dataset == "panasonic_18650pf"
            and payload.get("epoch") == 514
            and resolved.get("run_mode") == "formal"
            and resolved.get("development_only") is False
            and resolved.get("epochs") == 515
            and resolved.get("random_seed") == SEED
        ):
            return {
                "run_dir": run_dir, "checkpoint": checkpoint, "scaler": scaler,
                "config": config, "metadata": metadata,
            }
    return None

CANONICAL_RUN = discover_canonical_run(ARTIFACT_ROOTS)
CHECKPOINT_PATH = CANONICAL_RUN["checkpoint"] if CANONICAL_RUN else None
SCALER_PATH = CANONICAL_RUN["scaler"] if CANONICAL_RUN else None
CONFIG_PATH = CANONICAL_RUN["config"] if CANONICAL_RUN else None

manifest = {
    "method": "vidal_lstm_supervised_transfer",
    "paper": "Vidal et al. (2019)",
    "source_dataset": "panasonic_18650pf",
    "target_dataset": "cmu_evtol_vtc6",
    "source_case": 1,
    "source_temperature": "25_degC",
    "features": list(FEATURES),
    "target": "soc_dynamic_reference_not_ground_truth",
    "sampling_interval": "1_s",
    "sequence_length": 50,
    "stride": 50,
    "checkpoint_path": str(CHECKPOINT_PATH) if CHECKPOINT_PATH else None,
    "scaler_path": str(SCALER_PATH) if SCALER_PATH else None,
    "config_path": str(CONFIG_PATH) if CONFIG_PATH else None,
    "random_seed": SEED,
    "device": str(DEVICE),
    "data_version": "cmu_dynamic_capacity_soc_development_v1 + extension4",
    "git_commit": GIT_COMMIT,
    "timestamp": NOW_UTC,
    "implemented_sections": [0, 1, 2, 3, 4, 5, 7, 10, 11],
}
environment = {
    "python_version": platform.python_version(),
    "pytorch_version": torch.__version__,
    "device": str(DEVICE),
    "cuda_available": torch.cuda.is_available(),
    "git_commit": GIT_COMMIT,
    "timestamp_utc": NOW_UTC,
    "repo_root": str(REPO_ROOT),
    "checkpoint_exists": bool(CHECKPOINT_PATH and CHECKPOINT_PATH.is_file()),
    "scaler_exists": bool(SCALER_PATH and SCALER_PATH.is_file()),
    "run_config_exists": bool(CONFIG_PATH and CONFIG_PATH.is_file()),
    "repository_case1_config_exists": REPOSITORY_CASE1_CONFIG.is_file(),
    "cmu_inventory_exists": CMU_INVENTORY_PATH.is_file(),
    "external_artifact_root": str(EXTERNAL_ARTIFACT_ROOT) if EXTERNAL_ARTIFACT_ROOT else "NOT_AVAILABLE",
}
display(pd.DataFrame(environment.items(), columns=["item", "value"]))
display(manifest)

# 1. Method Definition

Vidal LSTM consumes voltage, current, and temperature sequences and outputs SOC at every time step. The reproduced architecture is one LSTM layer (hidden size 10) followed by a linear 10→1 head. PyTorch's recurrent-side secondary bias is frozen at zero to preserve the paper-aligned 571 trainable parameters.

- **Case 1:** supervised within-dataset source training/evaluation.
- **Case 2:** Case 1 source weights → supervised target full-train fine-tuning.
- **Case 3:** Case 1 source weights → supervised target partial-train fine-tuning.
- **Case 4:** reconstructed multi-source partial pretraining followed by supervised target full-train fine-tuning.

The method uses real target reference SOC during transfer and no pseudo labels. Each paper method may retain its own source-training procedure. The CMU benchmark standardizes target test cells, isolation rules, reference-label version, and metrics—not source-domain training.

In [ ]:
model_config = yaml.safe_load((REPO_ROOT / "configs/reproductions/vidal_2019/model.yaml").read_text()) or {}
case1_config = yaml.safe_load(REPOSITORY_CASE1_CONFIG.read_text()) or {}
method_table = pd.DataFrame([
    ("feature order", list(FEATURES)),
    ("sequence length", case1_config.get("seq_len", "NOT_CONFIRMED")),
    ("sampling interval", 1.0 / case1_config["resample_hz"] if case1_config.get("resample_hz") else "NOT_CONFIRMED"),
    ("stride", case1_config.get("stride", "NOT_CONFIRMED")),
    ("hidden size", model_config.get("hidden_size", "NOT_CONFIRMED")),
    ("LSTM layers", model_config.get("num_layers", "NOT_CONFIRMED")),
    ("dropout", model_config.get("dropout", "NOT_CONFIRMED")),
    ("optimizer", "Adam (Case 1 code)"),
    ("learning rate", case1_config.get("learning_rate", "NOT_CONFIRMED")),
    ("batch size", case1_config.get("batch_size", "NOT_CONFIRMED")),
    ("loss", "masked MSE (Case 1 code)"),
], columns=["setting", "value"])
display(method_table)

# 2. Original-Paper Reproduction: Cases 1–4

Existing artifacts are inventoried only; nothing is retrained. Paper-reproduction metrics remain separate from CMU benchmark metrics. A legacy Case 4 config still says “NOT YET IMPLEMENTED” even though the current Case 4 implementation exists; this is reported as an audit warning rather than silently corrected here.

In [ ]:
CASE_CONFIGS = {
    1: REPOSITORY_CASE1_CONFIG,
    2: REPO_ROOT / "configs/reproductions/vidal_2019/vidal_case2.yaml",
    3: REPO_ROOT / "configs/reproductions/vidal_2019/vidal_case3.yaml",
    4: REPO_ROOT / "configs/reproductions/vidal_2019/vidal_case4.yaml",
}
CASE_TYPES = {
    1: "supervised source training",
    2: "supervised full-target fine-tuning",
    3: "supervised partial-target fine-tuning",
    4: "multi-source pretraining + supervised target fine-tuning",
}

def identify_case(metadata, path):
    if metadata.get("case") in {2, 3, 4}:
        return int(metadata["case"])
    if metadata.get("dataset") in {"panasonic_18650pf", "lg_hg2", "samsung_30t"}:
        if "case2" not in str(path).lower() and "case3" not in str(path).lower() and "case4" not in str(path).lower():
            return 1
    return None

artifact_rows = []
for root in ARTIFACT_ROOTS:
    if not root.exists():
        continue
    for metadata_path in root.rglob("run_metadata.json"):
        try:
            metadata = json.loads(metadata_path.read_text())
        except Exception:
            continue
        case = identify_case(metadata, metadata_path)
        if case is None:
            continue
        run_dir = metadata_path.parent
        metrics_path = run_dir / "metrics.json"
        metrics = json.loads(metrics_path.read_text()) if metrics_path.is_file() else {}
        checkpoint = run_dir / "checkpoints/epoch_515.pt"
        scaler = run_dir / "scaler.json"
        config = run_dir / "resolved_config.yaml"
        artifact_rows.append({
            "case": case,
            "source dataset": metadata.get("source_dataset", metadata.get("dataset", "NOT_CONFIRMED") if case == 1 else "NOT_CONFIRMED"),
            "target dataset": metadata.get("target_dataset", "same_as_source" if case == 1 else "NOT_CONFIRMED"),
            "temperature": "25_degC" if case == 1 and metadata.get("dataset") == "panasonic_18650pf" else "NOT_CONFIRMED",
            "training/adaptation type": CASE_TYPES[case],
            "checkpoint": str(checkpoint) if checkpoint.is_file() else "NOT_AVAILABLE",
            "scaler": str(scaler) if scaler.is_file() else "NOT_AVAILABLE",
            "config": str(config) if config.is_file() else "NOT_AVAILABLE",
            "MAE": metrics.get("mae_fraction", metrics.get("mae", "NOT_AVAILABLE")),
            "RMSE": metrics.get("rmse_fraction", metrics.get("rmse", "NOT_AVAILABLE")),
            "same_run_artifacts": all(path.is_file() and path.parent == run_dir for path in (scaler, config))
                and checkpoint.is_file() and checkpoint.parent.parent == run_dir,
            "status": "AVAILABLE" if metrics_path.is_file() else "PARTIAL_ARTIFACT",
            "run_dir": str(run_dir),
        })

case_inventory_rows = []
for case in range(1, 5):
    matching = [row for row in artifact_rows if row["case"] == case]
    if matching:
        case_inventory_rows.extend(matching)
    else:
        case_inventory_rows.append({
            "case": case, "source dataset": "NOT_AVAILABLE", "target dataset": "NOT_AVAILABLE",
            "temperature": "NOT_AVAILABLE", "training/adaptation type": CASE_TYPES[case],
            "checkpoint": "NOT_AVAILABLE", "scaler": "NOT_AVAILABLE",
            "config": str(CASE_CONFIGS[case]) if CASE_CONFIGS[case].is_file() else "NOT_AVAILABLE",
            "MAE": "NOT_AVAILABLE", "RMSE": "NOT_AVAILABLE",
            "same_run_artifacts": False, "status": "NOT_AVAILABLE", "run_dir": "NOT_AVAILABLE",
        })
case_inventory = pd.DataFrame(case_inventory_rows)
display(case_inventory)

case4_text = CASE_CONFIGS[4].read_text() if CASE_CONFIGS[4].is_file() else ""
case4_artifact_exists = any(row["case"] == 4 and row["status"] == "AVAILABLE" for row in artifact_rows)
reproduction_audit_warnings = []
if "NOT YET IMPLEMENTED" in case4_text:
    reproduction_audit_warnings.append(
        "Legacy vidal_case4.yaml says NOT YET IMPLEMENTED while current case4.py is implemented"
        + (" and formal artifacts were found." if case4_artifact_exists else ".")
    )
display(pd.DataFrame({"audit_warning": reproduction_audit_warnings or ["none"]}))

paper_comparison = pd.DataFrame([{
    "case": case,
    "paper_MAE": "NOT_CONFIRMED",
    "paper_RMSE": "NOT_CONFIRMED",
    "reproduction_MAE": next((row["MAE"] for row in artifact_rows if row["case"] == case), "NOT_AVAILABLE"),
    "reproduction_RMSE": next((row["RMSE"] for row in artifact_rows if row["case"] == case), "NOT_AVAILABLE"),
} for case in range(1, 5)])
display(paper_comparison)
print("RUN_SOURCE_TRAINING =", RUN_SOURCE_TRAINING, "— source retraining is outside this notebook.")

# 3. Canonical Source Checkpoint and Scaler

The only accepted source pair is Panasonic 25 °C Case 1 `checkpoints/epoch_515.pt` plus sibling `scaler.json` and `resolved_config.yaml` from that same completed formal run. Case 2 scalers and cross-directory artifact mixing are rejected.

In [ ]:
MODEL = None
SCALER = None
CHECKPOINT = None
SOURCE_SYNTHETIC_STATUS = "NOT_RUN"
source_inventory = {
    "checkpoint_status": "NOT_AVAILABLE",
    "scaler_status": "NOT_AVAILABLE",
    "config_status": "NOT_AVAILABLE",
    "same_run_directory": "NOT_RUN",
    "checkpoint_config_match": "NOT_RUN",
    "scaler_matches_checkpoint_state": "NOT_RUN",
    "model_loaded": "NOT_RUN",
    "synthetic_inference": "NOT_RUN",
    "source_scaler_only": "NOT_RUN",
}

if CANONICAL_RUN:
    CHECKPOINT = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
    resolved = yaml.safe_load(CONFIG_PATH.read_text()) or {}
    scaler_json = json.loads(SCALER_PATH.read_text())
    checkpoint_config = CHECKPOINT.get("config", {})
    same_run = (
        CHECKPOINT_PATH.parent.parent == SCALER_PATH.parent
        == CONFIG_PATH.parent == CANONICAL_RUN["metadata"].parent
    )
    config_match = all(
        resolved.get(key) == checkpoint_config.get(key)
        for key in ("dataset", "seq_len", "stride", "random_seed", "epochs", "run_mode", "development_only")
    )
    MODEL = VidalLSTM().to(DEVICE)
    SCALER = MinMaxNormalizer()
    load_checkpoint(CHECKPOINT_PATH, MODEL, normalizer=SCALER, device=DEVICE)
    checkpoint_scaler = SCALER.state_dict()
    json_min = scaler_json.get("feature_min", scaler_json.get("min"))
    json_max = scaler_json.get("feature_max", scaler_json.get("max"))
    scaler_matches = (
        json_min is not None and json_max is not None
        and np.array_equal(np.asarray(json_min), np.asarray(checkpoint_scaler["min"]))
        and np.array_equal(np.asarray(json_max), np.asarray(checkpoint_scaler["max"]))
    )
    source_only = (
        scaler_json.get("dataset") == "panasonic_18650pf"
        and scaler_json.get("fit_split") == "full_train"
        and scaler_json.get("feature_names") == list(FEATURES)
    )
    if not (same_run and config_match and scaler_matches and source_only):
        raise RuntimeError("Canonical Case 1 checkpoint/scaler/config provenance validation failed")
    MODEL.eval()
    with torch.no_grad():
        synthetic = torch.zeros((2, 50, 3), dtype=torch.float32, device=DEVICE)
        synthetic_prediction = MODEL(synthetic).cpu().numpy()
    synthetic_ok = (
        synthetic_prediction.shape == (2, 50)
        and np.isfinite(synthetic_prediction).all()
        and CHECKPOINT.get("epoch") == 514
    )
    SOURCE_SYNTHETIC_STATUS = "PASS" if synthetic_ok else "FAIL"
    source_inventory.update({
        "checkpoint_status": "PASS",
        "checkpoint_size_bytes": CHECKPOINT_PATH.stat().st_size,
        "checkpoint_mtime_utc": datetime.fromtimestamp(CHECKPOINT_PATH.stat().st_mtime, timezone.utc).isoformat(),
        "checkpoint_keys": sorted(CHECKPOINT),
        "state_dict_parameter_shapes": {
            name: list(value.shape) for name, value in CHECKPOINT["model_state_dict"].items()
        },
        "checkpoint_epoch_zero_based": CHECKPOINT.get("epoch"),
        "input_dimension": MODEL.input_size,
        "sequence_length": resolved.get("seq_len"),
        "scaler_status": "PASS",
        "config_status": "PASS",
        "scaler_feature_order": scaler_json.get("feature_names", "NOT_CONFIRMED"),
        "scaler_feature_min": json_min,
        "scaler_feature_max": json_max,
        "scaler_fit_split": scaler_json.get("fit_split", "NOT_CONFIRMED"),
        "same_run_directory": "PASS" if same_run else "FAIL",
        "checkpoint_config_match": "PASS" if config_match else "FAIL",
        "scaler_matches_checkpoint_state": "PASS" if scaler_matches else "FAIL",
        "source_scaler_only": "PASS" if source_only else "FAIL",
        "model_loaded": "PASS",
        "synthetic_inference": SOURCE_SYNTHETIC_STATUS,
        "synthetic_input_shape": list(synthetic.shape),
        "synthetic_output_shape": list(synthetic_prediction.shape),
        "synthetic_prediction_min": float(synthetic_prediction.min()),
        "synthetic_prediction_max": float(synthetic_prediction.max()),
        "synthetic_nonfinite_count": int((~np.isfinite(synthetic_prediction)).sum()),
    })
display(pd.DataFrame(source_inventory.items(), columns=["check", "value"]))
if MODEL is None:
    print("NOT_AVAILABLE: set VIDAL_ARTIFACT_ROOT to a root containing the canonical formal Case 1 run.")

# 4. CMU Adapter and Input QC

The fixed core7 set is audited with the repository's real interfaces:

- `battery_benchmark.datasets.cmu_evtol.vidal_adapter.load_cmu_dynamic_soc_for_vidal()`
- `battery_benchmark.datasets.cmu_evtol.vidal_adapter.build_cmu_vidal_windows()`

No alternate loader, SOC calculator, resampler, or window builder is implemented here. The adapter calls `resample_vidal_record(..., clip_soc=False)`; the protocol config also sets `sampling.soc_clipping: false`.

In [ ]:
CORE7 = {
    "VAH01": ("baseline", CMU_BASE_ROOT / "VAH01_dynamic_soc.parquet"),
    "VAH17": ("baseline", CMU_BASE_ROOT / "VAH17_dynamic_soc.parquet"),
    "VAH27": ("baseline", CMU_BASE_ROOT / "VAH27_dynamic_soc.parquet"),
    "VAH30": ("35_degC", CMU_EXTENSION_ROOT / "VAH30_dynamic_soc.parquet"),
    "VAH15": ("1000_s_cruise", CMU_EXTENSION_ROOT / "VAH15_dynamic_soc.parquet"),
    "VAH12": ("400_s_cruise", CMU_EXTENSION_ROOT / "VAH12_dynamic_soc.parquet"),
    "VAH11": ("20_percent_lower_discharge_power", CMU_EXTENSION_ROOT / "VAH11_dynamic_soc.parquet"),
}
adapter_protocol_path = REPO_ROOT / "configs/datasets/cmu_evtol/protocols/cmu_evtol_vidal_adapter_v1.yaml"
adapter_protocol = yaml.safe_load(adapter_protocol_path.read_text()) or {}
committed_inventory = pd.read_csv(CMU_INVENTORY_PATH) if CMU_INVENTORY_PATH.is_file() else pd.DataFrame()

inventory_rows, feature_stats = [], []
for cell_id, (condition, path) in CORE7.items():
    frozen = committed_inventory.loc[committed_inventory["cell_id"] == cell_id]
    frozen = frozen.iloc[0].to_dict() if len(frozen) else {}
    row = {
        "cell_id": cell_id, "condition": condition,
        "file_path": str(path.relative_to(REPO_ROOT)) if path.is_file() else "NOT_AVAILABLE",
        "number_of_missions": frozen.get("vidal_records", "NOT_AVAILABLE"),
        "number_of_windows": frozen.get("vidal_windows", "NOT_AVAILABLE"),
        "sampling_interval": "1_s",
        "sequence_length": 50,
        "feature_columns": list(FEATURES),
        "target_column": "soc_dynamic",
        "SOC_min": frozen.get("soc_min", "NOT_AVAILABLE"),
        "SOC_max": frozen.get("soc_max", "NOT_AVAILABLE"),
        "has_nan": bool(frozen.get("nonfinite_count", 1)) if frozen else "NOT_AVAILABLE",
        "has_inf": bool(frozen.get("nonfinite_count", 1)) if frozen else "NOT_AVAILABLE",
    }
    inventory_rows.append(row)
    if path.is_file():
        raw = pd.read_parquet(path, columns=["voltage_v", "current_a", "Temperature__C", "soc_dynamic"])
        for source, feature in (("voltage_v", "voltage"), ("current_a", "current"), ("Temperature__C", "temperature")):
            values = raw[source].to_numpy(float)
            feature_stats.append({
                "cell_id": cell_id, "feature": feature,
                "min": float(values.min()), "max": float(values.max()),
                "mean": float(values.mean()), "std": float(values.std()),
                "nonfinite": int((~np.isfinite(values)).sum()),
            })
cmu_inventory = pd.DataFrame(inventory_rows)
cmu_feature_stats = pd.DataFrame(feature_stats)
display(cmu_inventory)
display(cmu_feature_stats)

In [ ]:
# Real adapter/window audit, processed one cell at a time.
deep_qc_rows = []
for cell_id, (_, path) in CORE7.items():
    if not path.is_file():
        deep_qc_rows.append({"cell_id": cell_id, "status": "NOT_AVAILABLE"})
        continue
    adapted = load_cmu_dynamic_soc_for_vidal([path], target_hz=1.0)
    windows = build_cmu_vidal_windows(adapted.records, seq_len=50, stride=50, keep_incomplete=False)
    by_order = {int(record.metadata["sequence_order"]): record for record in adapted.records}
    time_failures = sum(
        int(np.any(np.diff(record.data["timestamp"].to_numpy(float)) <= 0))
        for record in adapted.records
    )
    cross_mission = 0
    target_last_mismatch = 0
    keys = []
    for index, meta in enumerate(windows.metadata):
        record = by_order[int(meta["sequence_order"])]
        start, end = int(meta["window_start"]), int(meta["window_end"])
        if meta["mission_cycle_id"] != record.metadata["mission_cycle_id"] or end > len(record.data):
            cross_mission += 1
        expected_target = float(record.data["soc"].iloc[end - 1])
        if not np.array_equal(np.asarray(windows.y[index, int(meta["valid_length"]) - 1]), np.asarray(expected_target, dtype=np.float32)):
            target_last_mismatch += 1
        keys.append((cell_id, meta["mission_cycle_id"], start, end))
    soc_values = np.concatenate([record.data["soc"].to_numpy(float) for record in adapted.records])
    soc_unit = "fraction_0_1" if soc_values.min() >= 0 and soc_values.max() <= 1 else (
        "percent_0_100" if soc_values.min() >= 0 and soc_values.max() <= 100 else "NOT_CONFIRMED"
    )
    deep_qc_rows.append({
        "cell_id": cell_id, "status": "PASS",
        "records": len(adapted.records), "windows": len(windows.X),
        "window_shape": list(windows.X.shape[1:]),
        "time_order_failure_count": time_failures,
        "cross_mission_window_count": cross_mission,
        "target_last_mismatch_count": target_last_mismatch,
        "duplicate_window_count": len(keys) - len(set(keys)),
        "soc_unit": soc_unit,
        "soc_min": float(soc_values.min()), "soc_max": float(soc_values.max()),
        "nonfinite_count": int((~np.isfinite(windows.X)).sum() + (~np.isfinite(windows.y)).sum()),
        "soc_clipping_count": int(adapted.inventory["soc_clipping_count"].sum()),
    })
    del windows, adapted
deep_qc = pd.DataFrame(deep_qc_rows)
display(deep_qc)

assert (deep_qc["status"] == "PASS").all()
for column in (
    "time_order_failure_count", "cross_mission_window_count", "target_last_mismatch_count",
    "duplicate_window_count", "nonfinite_count", "soc_clipping_count",
):
    assert int(deep_qc[column].sum()) == 0, column
assert set(deep_qc["soc_unit"]) == {"fraction_0_1"}
assert all(shape == [50, 3] for shape in deep_qc["window_shape"])
assert adapter_protocol["sampling"]["soc_clipping"] is False

adapter_qc = {
    "adapter_version": ADAPTER_VERSION,
    "feature_order": list(FEATURES),
    "feature_order_match": tuple(FEATURES) == ("voltage", "current", "temperature"),
    "cross_mission_window_count": int(deep_qc["cross_mission_window_count"].sum()),
    "target_last_mismatch_count": int(deep_qc["target_last_mismatch_count"].sum()),
    "duplicate_window_count": int(deep_qc["duplicate_window_count"].sum()),
    "nonfinite_count": int(deep_qc["nonfinite_count"].sum()),
    "soc_unit": "fraction_0_1",
    "soc_clipping_enabled": False,
    "soc_clipping_count": int(deep_qc["soc_clipping_count"].sum()),
    "clipping_evidence": "vidal_adapter._resample_cmu_record -> resample_vidal_record(clip_soc=False); protocol sampling.soc_clipping=false",
    "cmu_scaler_refit": False,
}
display(pd.DataFrame(adapter_qc.items(), columns=["check", "value"]))

In [ ]:
# Three allowed diagnostics only.
vah01_path = CORE7["VAH01"][1]
if vah01_path.is_file():
    vah01 = pd.read_parquet(vah01_path)
    mission_id = vah01["mission_cycle_id"].iloc[0]
    mission = vah01[vah01["mission_cycle_id"] == mission_id]
    fig, axes = plt.subplots(4, 1, figsize=(11, 8), sharex=True)
    for ax, column, label in zip(
        axes,
        ["voltage_v", "current_a", "Temperature__C", "soc_dynamic"],
        ["Voltage (V)", "Current (A)", "Temperature (°C)", "Reference SOC"],
    ):
        ax.plot(mission["elapsed_candidate_s"], mission[column], linewidth=0.9)
        ax.set_ylabel(label)
    axes[-1].set_xlabel("Mission elapsed time (s)")
    fig.suptitle(f"VAH01 representative development candidate: {mission_id}")
    fig.tight_layout()
    plt.show()

SCALER_RANGE_QC_STATUS = "NOT_RUN"
transformed_stats = pd.DataFrame()
range_comparison = pd.DataFrame()
if SCALER is None:
    print("NOT_RUN: Panasonic scaler range, out-of-range ratios, and transformed distributions require canonical artifacts.")
else:
    state = SCALER.state_dict()
    source_ranges = pd.DataFrame({
        "feature": list(FEATURES), "source_min": state["min"], "source_max": state["max"]
    })
    range_comparison = cmu_feature_stats.groupby("feature", as_index=False).agg(
        cmu_min=("min", "min"), cmu_max=("max", "max")
    ).merge(source_ranges, on="feature")
    display(range_comparison)
    rows = []
    for cell_id, (_, path) in CORE7.items():
        raw = pd.read_parquet(path, columns=["voltage_v", "current_a", "Temperature__C"])
        x = raw.rename(columns={
            "voltage_v": "voltage", "current_a": "current", "Temperature__C": "temperature"
        })[list(FEATURES)].to_numpy(np.float32)
        transformed = SCALER.transform(x)
        for index, feature in enumerate(FEATURES):
            values = transformed[:, index]
            rows.append({
                "cell_id": cell_id, "feature": feature,
                "min": float(values.min()), "max": float(values.max()),
                "mean": float(values.mean()), "std": float(values.std()),
                "outside_source_range_fraction": float(np.mean((values < 0) | (values > 1))),
            })
    transformed_stats = pd.DataFrame(rows)
    display(transformed_stats)
    transformed_stats.boxplot(column="mean", by="feature")
    plt.suptitle("")
    plt.title("Scaler-transformed CMU feature means")
    plt.show()
    SCALER_RANGE_QC_STATUS = "PASS"

# 5. Zero-Shot Evaluation

Strict zero-shot means canonical Panasonic Case 1 checkpoint + frozen Panasonic Case 1 scaler + no CMU parameter update + no CMU scaler fitting + direct CMU inference. The smoke test uses VAH01 and at most three candidates, never writes formal results, and evaluates each window at its final target time point.

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

def state_hash(state):
    digest = hashlib.sha256()
    for name in sorted(state):
        value = state[name]
        digest.update(name.encode())
        digest.update(value.detach().cpu().contiguous().numpy().tobytes())
    return digest.hexdigest()

def json_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(",", ":")).encode()).hexdigest()

def detect_soc_unit(values):
    values = np.asarray(values, dtype=float)
    if np.isfinite(values).all() and values.min() >= 0 and values.max() <= 1:
        return "fraction_0_1"
    if np.isfinite(values).all() and values.min() >= 0 and values.max() <= 100:
        return "percent_0_100"
    return "NOT_CONFIRMED"

def evaluate_zero_shot_cell(cell_id, max_missions=None):
    if MODEL is None or SCALER is None:
        raise RuntimeError("Canonical Panasonic Case 1 checkpoint/scaler pair is unavailable")
    path = CORE7[cell_id][1]
    frame = pd.read_parquet(path)
    if max_missions is not None:
        selected = frame["mission_cycle_id"].drop_duplicates().iloc[:max_missions]
        frame = frame[frame["mission_cycle_id"].isin(selected)].copy()
        adapted = adapt_cmu_dynamic_soc(frame, source_path=str(path), target_hz=1.0)
    else:
        adapted = load_cmu_dynamic_soc_for_vidal([path], target_hz=1.0)
    transformed = transform_cmu_records(adapted.records, SCALER)
    windows = build_cmu_vidal_windows(transformed, seq_len=50, stride=50, keep_incomplete=False)
    dataset = TensorDataset(
        torch.from_numpy(windows.X), torch.from_numpy(windows.y), torch.arange(len(windows.X))
    )
    rows = []
    MODEL.eval()
    with torch.no_grad():
        for x, y, indices in DataLoader(dataset, batch_size=512, shuffle=False):
            prediction = MODEL(x.to(DEVICE)).cpu().numpy()
            for batch_index, window_index in enumerate(indices.tolist()):
                meta = windows.metadata[window_index]
                target_index = int(meta["valid_length"]) - 1
                reference = float(y[batch_index, target_index])
                predicted = float(prediction[batch_index, target_index])
                rows.append({
                    "model": "VidalLSTM", "paper": "Vidal et al. (2019)",
                    "track": "zero_shot", "adaptation_type": "none", "fold": "source_only",
                    "seed": SEED, "source_dataset": "panasonic_18650pf",
                    "source_case": 1, "adaptation_cells": "", "test_cell": cell_id,
                    "condition": CORE7[cell_id][0], "target_labels_used": False,
                    "pseudo_labels_used": False, "split": "test", "phase": "NOT_AVAILABLE",
                    "mission_id": meta["mission_cycle_id"], "window_index": window_index,
                    "window_target_index": target_index,
                    "soc_reference": reference, "soc_prediction": predicted,
                    "absolute_error": abs(predicted - reference),
                })
    predictions = pd.DataFrame(rows)
    metric_rows = []
    for mission_id, group in predictions.groupby("mission_id", sort=False):
        metric = compute_metrics(group["soc_reference"], group["soc_prediction"])
        manual_mae = float(np.mean(np.abs(group["soc_prediction"] - group["soc_reference"])))
        manual_rmse = float(np.sqrt(np.mean((group["soc_prediction"] - group["soc_reference"]) ** 2)))
        assert np.isclose(metric["mae"], manual_mae) and np.isclose(metric["rmse"], manual_rmse)
        metric_rows.append({
            "model": "VidalLSTM", "paper": "Vidal et al. (2019)", "track": "zero_shot",
            "adaptation_type": "none", "fold": "source_only", "seed": SEED,
            "source_dataset": "panasonic_18650pf", "source_case": 1,
            "adaptation_cells": "", "test_cell": cell_id, "condition": CORE7[cell_id][0],
            "target_labels_used": False, "pseudo_labels_used": False, "split": "test",
            "phase": "all", "mission_id": mission_id,
            "n_windows": int(group["window_index"].nunique()),
            "mae": metric["mae"], "rmse": metric["rmse"],
            "checkpoint_path": str(CHECKPOINT_PATH), "scaler_path": str(SCALER_PATH),
            "config_path": str(CONFIG_PATH), "data_version": manifest["data_version"],
            "git_commit": GIT_COMMIT, "timestamp": NOW_UTC,
        })
    return predictions, pd.DataFrame(metric_rows)

zero_shot_predictions = pd.DataFrame()
zero_shot_mission_metrics = pd.DataFrame()
zero_shot_cell_metrics = pd.DataFrame()
ZERO_SHOT_SMOKE_STATUS = "NOT_RUN"
FULL_ZERO_SHOT_STATUS = "NOT_RUN"
ZERO_SHOT_QC = {}

if RUN_ZERO_SHOT_SMOKE_TEST:
    if MODEL is None or SCALER is None:
        print("NOT_RUN: canonical checkpoint/scaler/config artifacts are unavailable.")
    else:
        model_hash_before = state_hash(MODEL.state_dict())
        scaler_hash_before = json_hash(SCALER.state_dict())
        zero_shot_predictions, zero_shot_mission_metrics = evaluate_zero_shot_cell("VAH01", 3)
        model_hash_after = state_hash(MODEL.state_dict())
        scaler_hash_after = json_hash(SCALER.state_dict())
        reference_unit = detect_soc_unit(zero_shot_predictions["soc_reference"])
        prediction_unit = detect_soc_unit(zero_shot_predictions["soc_prediction"])
        ZERO_SHOT_QC = {
            "data_loaded": not zero_shot_predictions.empty,
            "input_shape": [50, 3],
            "model_eval_mode": not MODEL.training,
            "reference_soc_unit": reference_unit,
            "prediction_soc_unit": prediction_unit,
            "units_match": reference_unit == prediction_unit,
            "nonfinite_count": int((~np.isfinite(
                zero_shot_predictions[["soc_reference", "soc_prediction"]]
            )).sum().sum()),
            "prediction_target_order_preserved": zero_shot_predictions["window_index"].is_monotonic_increasing,
            "model_parameter_hash_unchanged": model_hash_before == model_hash_after,
            "scaler_state_hash_unchanged": scaler_hash_before == scaler_hash_after,
            "cmu_scaler_fit_called": False,
            "n_missions": int(zero_shot_predictions["mission_id"].nunique()),
            "n_windows": len(zero_shot_predictions),
        }
        if not all([
            ZERO_SHOT_QC["data_loaded"], ZERO_SHOT_QC["model_eval_mode"],
            ZERO_SHOT_QC["units_match"], ZERO_SHOT_QC["nonfinite_count"] == 0,
            ZERO_SHOT_QC["prediction_target_order_preserved"],
            ZERO_SHOT_QC["model_parameter_hash_unchanged"],
            ZERO_SHOT_QC["scaler_state_hash_unchanged"],
        ]):
            raise RuntimeError("Zero-shot smoke QC failed")
        ZERO_SHOT_SMOKE_STATUS = "PASS"
        display(pd.DataFrame(ZERO_SHOT_QC.items(), columns=["check", "value"]))
        display(zero_shot_mission_metrics)

if RUN_FULL_ZERO_SHOT:
    if MODEL is None or SCALER is None:
        print("NOT_RUN: full zero-shot requires canonical source artifacts.")
    else:
        predictions, metrics = [], []
        for cell_id in CORE7:
            cell_predictions, cell_metrics = evaluate_zero_shot_cell(cell_id)
            predictions.append(cell_predictions)
            metrics.append(cell_metrics)
        zero_shot_predictions = pd.concat(predictions, ignore_index=True)
        zero_shot_mission_metrics = pd.concat(metrics, ignore_index=True)
        FULL_ZERO_SHOT_STATUS = "PASS"

if not zero_shot_mission_metrics.empty:
    zero_shot_cell_metrics = zero_shot_mission_metrics.groupby(
        ["model", "paper", "track", "test_cell", "condition"], as_index=False
    ).agg(
        n_missions=("mission_id", "nunique"),
        mission_macro_mae=("mae", "mean"),
        mission_macro_rmse=("rmse", "mean"),
    )
    display(zero_shot_cell_metrics)

In [ ]:
if not zero_shot_predictions.empty:
    mission_id = zero_shot_predictions["mission_id"].iloc[0]
    plot_frame = zero_shot_predictions[zero_shot_predictions["mission_id"] == mission_id]
    fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
    axes[0].plot(plot_frame["soc_reference"].to_numpy(), label="reference")
    axes[0].plot(plot_frame["soc_prediction"].to_numpy(), label="prediction")
    axes[0].legend()
    axes[0].set_ylabel("SOC fraction")
    axes[0].set_title("track=zero_shot | target_labels_used=false | pseudo_labels_used=false")
    axes[1].plot(plot_frame["absolute_error"].to_numpy(), color="tab:red")
    axes[1].set_ylabel("Absolute error")
    axes[1].set_xlabel("Ordered window target")
    fig.tight_layout()
    plt.show()

# 6. Vidal Original Transfer Protocol

**STATUS = NOT_IMPLEMENTED for CMU.**

The repository confirms supervised target reference SOC, all 571 trainable parameters updated (architecture-equivalence bias remains frozen), target-scaler refitting in Cases 2–4, Adam, learning rate 0.01, and 515 formal epochs. The exact original-paper target quantity and any validation/early-stopping details not encoded in reviewed artifacts remain `NOT_CONFIRMED`.

No CMU label ratio, mission split, validation ratio, freeze policy, scaler policy, early stopping, or hyperparameter-selection policy is chosen here.

In [ ]:
transfer_items = pd.DataFrame([
    ("target data quantity", "NOT_CONFIRMED"),
    ("true target reference SOC", True),
    ("updated parameters", "all 571 trainable parameters in current Case 2–4 code"),
    ("frozen parameters", "40-element PyTorch secondary LSTM bias for architecture equivalence"),
    ("scaler strategy in Case 2–4", "target scaler refit"),
    ("optimizer", "Adam"),
    ("learning rate", 0.01),
    ("formal epochs", 515),
    ("early stopping", "NOT_CONFIRMED"),
    ("training/validation/test split", "NOT_CONFIRMED"),
], columns=["item", "confirmed value"])
cmu_adaptation_interface = {
    "STATUS": "NOT_IMPLEMENTED",
    "target_label_ratio": "NOT_FROZEN",
    "mission_split": "NOT_FROZEN",
    "validation_ratio": "NOT_FROZEN",
    "layer_freezing": "NOT_FROZEN",
    "scaler_policy": "NOT_FROZEN",
    "early_stopping": "NOT_FROZEN",
    "hyperparameter_selection": "NOT_FROZEN",
}
if RUN_SUPERVISED_ADAPTATION:
    raise RuntimeError("BLOCKED_BY_PROTOCOL: CMU supervised adaptation protocol is not frozen")
display(transfer_items)
display(cmu_adaptation_interface)

# 7. CMU Baseline Three-Fold Supervised Adaptation

Only a fold manifest and fail-closed isolation assertions are produced. Adaptation uses true CMU reference SOC, never pseudo labels. Scaler policy remains `NOT_FROZEN`.

In [ ]:
baseline_cells = {"VAH01", "VAH17", "VAH27"}
folds = pd.DataFrame([
    {"fold": 1, "adaptation_cells": ["VAH17", "VAH27"], "held_out_test_cell": "VAH01"},
    {"fold": 2, "adaptation_cells": ["VAH01", "VAH27"], "held_out_test_cell": "VAH17"},
    {"fold": 3, "adaptation_cells": ["VAH01", "VAH17"], "held_out_test_cell": "VAH27"},
])
for _, row in folds.iterrows():
    adaptation = set(row["adaptation_cells"])
    held_out = {row["held_out_test_cell"]}
    assert adaptation.isdisjoint(held_out)
    assert adaptation | held_out == baseline_cells
    assert len(adaptation) == 2 and len(held_out) == 1
folds["held_out_in_training"] = False
folds["held_out_in_validation"] = False
folds["held_out_in_scaler_fit"] = False
folds["held_out_in_hyperparameter_selection"] = False
folds["target_labels_used"] = True
folds["pseudo_labels_used"] = False
folds["scaler_policy"] = "NOT_FROZEN"
folds["status"] = "BLOCKED_BY_PROTOCOL"
display(folds)

# 8. Condition-Cell Stress Tests

**STATUS = NOT_IMPLEMENTED.** Each future fold-adapted model will separately test VAH30, VAH15, VAH12, and VAH11; each condition will then report mean ± standard deviation across three fold models. Random weights, zero-shot models, and fabricated results are not substitutes.

# 9. Phase-Level Diagnostics

The requested assignment uses the phase at each window's final target time point, with take-off, cruise, and landing mission-macro MAE/RMSE. Current CMU Vidal adapter records do not expose a reviewed formal phase label using those names. Therefore zero-shot phase diagnostics are `NOT_RUN`, and supervised-adaptation phase diagnostics remain `NOT_IMPLEMENTED`; no development screening segment is silently renamed.

# 10. Standardized Result Export

The existing repository namespace is used: `outputs/benchmarks/panasonic_to_cmu_evtol/vidal/`. Writes are disabled by default. Smoke results, if explicitly written, remain under `_smoke` and cannot overwrite full results.

In [ ]:
RESULT_COLUMNS = [
    "model", "paper", "track", "adaptation_type", "fold", "seed", "source_dataset",
    "source_case", "adaptation_cells", "test_cell", "condition", "target_labels_used",
    "pseudo_labels_used", "split", "phase", "mission_id", "n_windows", "mae", "rmse",
    "checkpoint_path", "scaler_path", "config_path", "data_version", "git_commit", "timestamp",
]
export_status = "NOT_RUN"
if WRITE_RESULTS:
    if zero_shot_predictions.empty:
        print("NOT_RUN: no real zero-shot result exists to export.")
    else:
        scope = "full" if FULL_ZERO_SHOT_STATUS == "PASS" else "_smoke"
        output_dir = RESULT_ROOT / scope
        output_dir.mkdir(parents=True, exist_ok=True)
        zero_shot_predictions.to_parquet(output_dir / "zero_shot_predictions.parquet", index=False)
        zero_shot_mission_metrics.reindex(columns=RESULT_COLUMNS).to_csv(
            output_dir / "zero_shot_mission_metrics.csv", index=False
        )
        zero_shot_cell_metrics.to_csv(output_dir / "zero_shot_cell_metrics.csv", index=False)
        (output_dir / "experiment_manifest.json").write_text(json.dumps(manifest, indent=2) + "\n")
        qc = {
            "track": "zero_shot", "target_labels_used": False, "pseudo_labels_used": False,
            "smoke_status": ZERO_SHOT_SMOKE_STATUS, "full_status": FULL_ZERO_SHOT_STATUS,
            "source_scaler_range_qc": SCALER_RANGE_QC_STATUS, "zero_shot_qc": ZERO_SHOT_QC,
        }
        (output_dir / "qc_summary.json").write_text(json.dumps(qc, indent=2) + "\n")
        export_status = "PASS"
        print("Exported:", output_dir)
else:
    print("NOT_RUN: WRITE_RESULTS=False; no output was created or overwritten.")

# 11. Final Validation Summary

In [ ]:
def evidence_status(condition, unavailable="FAIL"):
    return "PASS" if condition else unavailable

case_found = {
    case: any(row["case"] == case and row["status"] == "AVAILABLE" for row in artifact_rows)
    for case in range(1, 5)
}
summary_rows = [
    *[(f"Case {case} reproduction results found", evidence_status(case_found[case], "NOT_RUN")) for case in range(1, 5)],
    ("Canonical Case 1 checkpoint found", evidence_status(CHECKPOINT_PATH is not None, "NOT_RUN")),
    ("Canonical Case 1 scaler found", evidence_status(SCALER_PATH is not None, "NOT_RUN")),
    ("Checkpoint and scaler belong to same run", source_inventory["same_run_directory"]),
    ("Model loaded successfully", source_inventory["model_loaded"]),
    ("Synthetic inference passed", source_inventory["synthetic_inference"]),
    ("Feature order matched", evidence_status(adapter_qc["feature_order_match"])),
    ("Sequence length matched", evidence_status(case1_config.get("seq_len") == 50)),
    ("CMU adapter files found", evidence_status(all(path.is_file() for _, path in CORE7.values()))),
    ("No CMU scaler refit", evidence_status(adapter_qc["cmu_scaler_refit"] is False)),
    ("No cross-mission windows", evidence_status(adapter_qc["cross_mission_window_count"] == 0)),
    ("No NaN/Inf", evidence_status(adapter_qc["nonfinite_count"] == 0)),
    ("SOC unit confirmed", evidence_status(adapter_qc["soc_unit"] == "fraction_0_1", "NOT_CONFIRMED")),
    ("VAH01 zero-shot smoke test passed", ZERO_SHOT_SMOKE_STATUS),
    ("Full zero-shot completed", FULL_ZERO_SHOT_STATUS),
    ("Supervised adaptation implemented", "NOT_IMPLEMENTED"),
    ("Condition stress test completed", "NOT_IMPLEMENTED"),
    ("Standardized export completed", export_status),
]
validation_summary = pd.DataFrame(summary_rows, columns=["check", "status"])
allowed = {"PASS", "FAIL", "NOT_RUN", "NOT_IMPLEMENTED", "NOT_CONFIRMED"}
assert set(validation_summary["status"]).issubset(allowed)
display(validation_summary)

print("CMU soc_dynamic remains a development reference label, not ground truth.")
print("No source training, CMU scaler fitting, supervised adaptation, pseudo-labeling, or result writing occurred by default.")